In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/attc-dataset/TestV2 - testV2.csv
/kaggle/input/attc-dataset/trainV2.csv


In [2]:
!pip install -q transformers datasets accelerate scikit-learn

In [3]:
import os
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


2026-02-12 16:21:21.926524: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770913282.150160      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770913282.215897      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770913282.765371      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770913282.765403      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770913282.765406      24 computation_placer.cc:177] computation placer alr

In [4]:
train_df = pd.read_csv("/kaggle/input/attc-dataset/trainV2.csv")
test_df  = pd.read_csv("/kaggle/input/attc-dataset/TestV2 - testV2.csv")

train_df = train_df.dropna().reset_index(drop=True)
test_df  = test_df.dropna().reset_index(drop=True)

label2id = {"Non-Abusive": 0, "Abusive": 1, "abusive":1}
id2label = {0: "Non-Abusive", 1: "Abusive"}

train_df["label"] = train_df["Class"].map(label2id)

In [5]:
train_df["label"].isna().sum()

np.int64(0)

In [6]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df["Text"].tolist(),
    train_df["label"].tolist(),
    test_size=0.15,
    stratify=train_df["label"],
    random_state=SEED
)

In [7]:
MODEL_NAME = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

In [8]:
class TamilAbuseDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels=None):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=128
        )
        if self.labels is not None:
            encoding["labels"] = self.labels[idx]
        return encoding

In [9]:
train_dataset = TamilAbuseDataset(train_texts, train_labels)
val_dataset   = TamilAbuseDataset(val_texts, val_labels)
test_dataset  = TamilAbuseDataset(test_df["Text"].tolist())

In [10]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0,1]),
    y=train_labels
)

class_weights = torch.tensor(class_weights, dtype=torch.float)

In [11]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

model.to("cuda")

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


XLMRobertaForSequenceClassification(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=

In [12]:
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        if class_weights is not None:
            self.class_weights = class_weights.to(self.model.device)
        else:
            self.class_weights = None

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        labels = inputs["labels"]
    
        outputs = model(**inputs)
        logits = outputs.logits
    
        # ✅ SAFE: works for DP / DDP / single GPU
        num_labels = self.model.config.num_labels
    
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(
            logits.view(-1, num_labels),
            labels.view(-1),
        )
    
        return (loss, outputs) if return_outputs else loss



In [13]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    macro_f1 = f1_score(labels, preds, average="macro")
    return {"macro_f1": macro_f1}

In [14]:
training_args = TrainingArguments(
    output_dir="./outputs",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    fp16=True,
    logging_steps=100,
    report_to="none",
    seed=SEED,
    warmup_ratio=0.1
)

In [15]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics
)


/tmp/ipykernel_24/1003253663.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


In [16]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.542375,0.729318
2,0.664500,0.644989,0.698637
3,0.509500,0.487153,0.775217
4,0.446500,0.496924,0.782811
5,0.360400,0.494095,0.795577


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


TrainOutput(global_step=485, training_loss=0.4651492718568782, metrics={'train_runtime': 313.038, 'train_samples_per_second': 49.579, 'train_steps_per_second': 1.549, 'total_flos': 828586046651520.0, 'train_loss': 0.4651492718568782, 'epoch': 5.0})

In [17]:
SAVE_DIR = "/kaggle/working/xlm_r_abuse_model_v2"

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

('/kaggle/working/xlm_r_abuse_model_v2/tokenizer_config.json',
 '/kaggle/working/xlm_r_abuse_model_v2/special_tokens_map.json',
 '/kaggle/working/xlm_r_abuse_model_v2/sentencepiece.bpe.model',
 '/kaggle/working/xlm_r_abuse_model_v2/added_tokens.json',
 '/kaggle/working/xlm_r_abuse_model_v2/tokenizer.json')

In [18]:
val_preds = trainer.predict(val_dataset)
y_true = val_labels
y_pred = np.argmax(val_preds.predictions, axis=1)

print(classification_report(
    y_true,
    y_pred,
    target_names=["Non-Abusive", "Abusive"],
    digits=4
))

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


              precision    recall  f1-score   support

 Non-Abusive     0.8327    0.7562    0.7926       283
     Abusive     0.7629    0.8377    0.7986       265

    accuracy                         0.7956       548
   macro avg     0.7978    0.7970    0.7956       548
weighted avg     0.7989    0.7956    0.7955       548



In [19]:
probs = torch.softmax(
    torch.tensor(val_preds.predictions),
    dim=1
).numpy()[:,1]

best_f1 = 0
best_thresh = 0.5

for t in np.arange(0.3, 0.7, 0.01):
    preds = (probs >= t).astype(int)
    f1 = f1_score(y_true, preds, average="macro")
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t

print("Best Threshold:", best_thresh)
print("Best Macro-F1:", best_f1)

Best Threshold: 0.5900000000000003
Best Macro-F1: 0.8029092071611253


In [20]:
test_preds = trainer.predict(test_dataset)
test_probs = torch.softmax(
    torch.tensor(test_preds.predictions),
    dim=1
).numpy()[:,1]

test_labels = (test_probs >= best_thresh).astype(int)

In [21]:
submission = pd.DataFrame({
    "Text": test_df["Text"],
    "Standard": [id2label[x] for x in test_labels]
})

submission.to_csv("submission.csv", index=False)